### To generate standard datasets
#### task demo gift box

### Step 1. convert segment ploygen points to rle type

In [1]:
import os
from pathlib import Path
os.chdir("../../")

print(os.getcwd())
import json
from pycocotools import mask as mask_utils

ori_annotation_path = Path("datasets/giftbox_task/annotations/annotations_0001.json")
## need to create file
rle_type_file_path = Path("datasets/giftbox_task/rle_annotations/annotations_0001.json")


def rle_convert(ori_annotation_path, rle_type_file_path):
    #rle_type_file_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(ori_annotation_path, 'r') as f:
        coco_data = json.load(f)

    images_map = {img['id']: img for img in coco_data['images']}
    converted_count = 0
    for ann in coco_data['annotations']:
        seg = ann.get('segmentation', [])
        
        if not seg or (isinstance(seg, dict) and 'counts' in seg and isinstance(seg['counts'], str)):
            continue
        
        img_info = images_map[ann['image_id']]
        height, width = img_info['height'], img_info['width']
        try:
            rles = mask_utils.frPyObjects(seg, height, width)
            rle = mask_utils.merge(rles)
            if isinstance(rle['counts'], bytes):
                rle['counts'] = rle['counts'].decode('utf-8')
            ann['segmentation'] = rle
            converted_count += 1
            
        except Exception as e:
            print(f"Warning: 转换标注 ID {ann.get('id')} 失败: {e}")
            continue
    
    with open(rle_type_file_path, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"{converted_count} save: {rle_type_file_path}")
    


/home/kewei/automatic-annotation


In [ ]:
rle_convert(ori_annotation_path,rle_type_file_path)



In [ ]:
## batch convert 
from tqdm import tqdm

origin_anno_dir = Path("datasets/giftbox_task/annotations")
rle_anno_dir = Path("datasets/giftbox_task/rle_annotations")

f_name = origin_anno_dir.glob("*.json")

for name in tqdm(f_name,desc="convert process:"):
    o_file = origin_anno_dir / name.name
    rle_file = rle_anno_dir / name.name

    rle_convert(o_file,rle_file)



## Step 2. make standard datasets structure

In [ ]:
## eg:

"""
"images":
[
    {
      "id": 0,
      "video_id":"0001",  
      "file_name": "1/video1/metaclip_1_1001_c122868928880ae52b33fae1.jpeg",
      "frame_id" : 1 ,
      "width": 600,
      "height": 600,
      "text_inputs": ["red box", "green small box", "small zip", "toy car"],
      "queried_categories": ["0", "2001", "0", "0"],
      "is_instance_exhaustive": 1,
      "is_pixel_exhaustive": 1,
      "scene_description" : "box free and closed",
      "task_type": "assembly task"
    },
]

"annotation":
[
   {
      "id": 1,
      "image_id": 0,
      "text_idx": 0,
      "source": "manual",
      "area": 0.00248,
      "bbox": [0.44333, 0.0, 0.10833, 0.05833],
      "segmentation": {
       ##use rle type "counts": "`kk42fb01O1O1O1O001O1O1O001O1O00001O1O001O001O0000000000O1001000O010O02O001N10001N0100000O10O1000O10O010O100O1O1O1O1O0000001O0O2O1N2N2Nobm4",
        "size": [600, 600]
      },
      "category_id": 1,
      "iscrowd": 0
    },
]
"""

In [2]:
from pathlib import Path
label_dir = Path("datasets/giftbox_task/label")
import numpy as np


def get_description(video_id,frame_idx):

    des_dict = {
    "1": "box free and closed",
    "2": "box fixed and closed",
    "3": "box fixed and opening",
    "4": "box open and toy free",
    "5": "toy in hand and outside box",
    "6": "toy in box and box opened",
    "7": "toy in box and box closing",
    "8": "toy in box and box on table",
    "9": "right hand holding red bag, left hand holding green box",
    "10": "toy in box and box in hand",
    "11": "gift has been packaged"
}   
    label_f = label_dir / (str(video_id)+"_label.txt")


    matrix = np.loadtxt(label_f, dtype=int)

    cls_id = str(matrix[frame_idx][1])

    return des_dict[cls_id]




In [3]:
def get_instance_annostatus(image_id,annotation_list,categorys):
    cat_id_to_name = {cat['id']: cat['name'] for cat in categorys}
    text_inputs = []
    queried_categories = []

    for ann in annotation_list:
        if ann['image_id'] == image_id:
            cat_id = ann['category_id']
            queried_categories.append(str(cat_id))
            text_inputs.append(cat_id_to_name.get(cat_id, str(cat_id)))

    return text_inputs,queried_categories


In [4]:
def build_annotation_field(image_id, annotation_list):
    bboxs = []
    segmentations = []
    total_area = 0
    queried_categories = []

    for ann in annotation_list:
        if ann['image_id'] == image_id:
            bboxs.append(ann['bbox'])
            segmentations.append(ann['segmentation'])
            queried_categories.append(str(ann['category_id']))
            total_area += ann['area']

    if not bboxs:
        return None

    return {
        "id": image_id,
        "image_id": image_id,
        "source": "automatic",
        "area": total_area,
        "bboxs": bboxs,
        "segmentations": segmentations,
        "queried_categories": queried_categories,
        "iscrowd": 0
    }


In [5]:
## first build images json 
import re
import json
from pathlib import Path

rle_annotation_file = Path("datasets/giftbox_task/rle_annotations/annotations_0001.json")

def build_image_field(rle_annotation_file):

    with open(rle_annotation_file, 'r') as f:
        coco_data = json.load(f)

    images_list = coco_data['images']
    annotation_list = coco_data['annotations']
    categorys = coco_data['categories']

    match = re.search(r"annotations_(\d+)\.json", rle_annotation_file.name)
    if match:
        video_id = match.group(1)
    else:
        video_id = "-1"

    label_f = label_dir / (str(video_id) + "_label.txt")
    if label_f.exists():
        matrix = np.loadtxt(label_f, dtype=int)
        if 12 in matrix[:, 1] or 0 in matrix[:, 1]:
            print(f"Skip {rle_annotation_file.name}: class 0 or 12 found")
            return None

    target_image_fieid = []
    target_anno_fieid = []

    for img in images_list:

        frame_idx_raw = img.get("frame_idx", "")
        match_frameid = re.search(r"frame_(\d+)", str(frame_idx_raw))
        if match_frameid:
            frame_idx_int = int(match_frameid.group(1))
        else:
            try:
                frame_idx_int = int(frame_idx_raw)
            except (ValueError, TypeError):
                frame_idx_int = -1

        text_inputs, queried_categories = get_instance_annostatus(img["id"], annotation_list, categorys)

        target_dict = {
            "id": img["id"],
            "video_id": str(video_id),
            "file_name": str(video_id) + "/" + img["file_name"],
            "frame_idx": img.get("frame_idx", ""),
            "width": img["width"],
            "height": img["height"],
            "text_inputs": text_inputs,
            "queried_categories": queried_categories,
            "is_instance_exhaustive": 1,
            "is_pixel_exhaustive": 1,
            "scene_description": get_description(video_id, frame_idx_int),
            "task_type": "assembly task"
        }

        target_image_fieid.append(target_dict)

        anno = build_annotation_field(img["id"], annotation_list)
        if anno is not None:
            target_anno_fieid.append(anno)

    result = {
        "images": target_image_fieid,
        "annotations": target_anno_fieid,
        "categories":coco_data['categories']
    }

    output_dir = rle_annotation_file.parent.parent / "standard_dataset"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / rle_annotation_file.name
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=2)

    print(f"Saved: {output_file}")
    return result



In [11]:

build_image_field(rle_annotation_file)



Saved: datasets/giftbox_task/standard_dataset/annotations_0001.json


{'images': [{'id': 1,
   'video_id': '0001',
   'file_name': '0001/frame_00000.jpg',
   'frame_idx': 0,
   'width': 640,
   'height': 480,
   'text_inputs': ['red box',
    'small object',
    'small object',
    'green small box',
    'toy car'],
   'queried_categories': ['0', '1', '1', '2', '3'],
   'is_instance_exhaustive': 1,
   'is_pixel_exhaustive': 1,
   'scene_description': 'box free and closed',
   'task_type': 'assembly task'},
  {'id': 2,
   'video_id': '0001',
   'file_name': '0001/frame_00015.jpg',
   'frame_idx': 15,
   'width': 640,
   'height': 480,
   'text_inputs': ['red box',
    'small object',
    'small object',
    'green small box',
    'toy car'],
   'queried_categories': ['0', '1', '1', '2', '3'],
   'is_instance_exhaustive': 1,
   'is_pixel_exhaustive': 1,
   'scene_description': 'box free and closed',
   'task_type': 'assembly task'},
  {'id': 3,
   'video_id': '0001',
   'file_name': '0001/frame_00030.jpg',
   'frame_idx': 30,
   'width': 640,
   'height': 

### Step3. Batch to process

In [6]:
from tqdm import tqdm

standard_dir = Path("datasets/giftbox_task/standard_dataset")

rle_annotation_dir = Path("datasets/giftbox_task/rle_annotations")

rle_list = rle_annotation_dir.glob("*.json")

for rle_f in tqdm(rle_list,desc="processing :"):

    build_image_field(rle_f)

print("task done.")




processing :: 9it [00:00, 84.51it/s]

Saved: datasets/giftbox_task/standard_dataset/annotations_0147.json
Skip annotations_0098.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0072.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0073.json
Skip annotations_0059.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0177.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0182.json
Skip annotations_0060.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0038.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0151.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0138.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0156.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0043.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0157.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0002.json
Skip annotations_0070.json: class 0 or 1

processing :: 25it [00:00, 54.87it/s]

Saved: datasets/giftbox_task/standard_dataset/annotations_0134.json
Skip annotations_0084.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0153.json
Skip annotations_0027.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0112.json
Skip annotations_0054.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0166.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0173.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0023.json
Skip annotations_0093.json: class 0 or 12 found
Skip annotations_0083.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0047.json
Skip annotations_0127.json: class 0 or 12 found


processing :: 39it [00:00, 60.54it/s]

...

Saved: datasets/giftbox_task/standard_dataset/annotations_0115.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0053.json
Skip annotations_0121.json: class 0 or 12 found
Skip annotations_0130.json: class 0 or 12 found
Skip annotations_0032.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0160.json
Skip annotations_0051.json: class 0 or 12 found
Skip annotations_0126.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0107.json
Skip annotations_0013.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0108.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0114.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0174.json


processing :: 170it [00:02, 63.92it/s]

Saved: datasets/giftbox_task/standard_dataset/annotations_0116.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0117.json
Skip annotations_0036.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0044.json
Skip annotations_0042.json: class 0 or 12 found
Skip annotations_0014.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0136.json
Skip annotations_0085.json: class 0 or 12 found
Skip annotations_0021.json: class 0 or 12 found
Skip annotations_0056.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0154.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0161.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0045.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0006.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0155.json
Skip annotations_0078.json: class 0 or 12 found


processing :: 185it [00:02, 66.64it/s]

Saved: datasets/giftbox_task/standard_dataset/annotations_0135.json
Skip annotations_0144.json: class 0 or 12 found
Skip annotations_0033.json: class 0 or 12 found
Saved: datasets/giftbox_task/standard_dataset/annotations_0183.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0003.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0103.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0091.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0089.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0169.json
Saved: datasets/giftbox_task/standard_dataset/annotations_0071.json
Skip annotations_0037.json: class 0 or 12 found
Skip annotations_0086.json: class 0 or 12 found
task done.
